In [1]:
import torch 
USE_CUDA = torch.cuda.is_available()
device = torch.device('cuda:0' if USE_CUDA else 'cpu')

In [2]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import ffmpeg as ff
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
import pickle
import multiprocessing
import datetime
from tqdm import tqdm
from facenet_pytorch import MTCNN
import logging
from einops import rearrange
import logging

from torchvision import transforms, utils, models
from PIL import Image
# import face_recognition

In [3]:
import logging

logging.basicConfig(
    filename='./LOG/random_user.log',  # 日志输出文件
    filemode='a',
    # level=logging.DEBUG,
    format='[%(asctime)s]::%(process)d::%(levelname)s::%(message)s'
)
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
# fileHandler = logging.FileHandler('./LOG/personalityLog.log')
# fileHandler.setFormatter(formatter)
# logger.addHandler(streamHandler)
# logger.addHandler(fileHandler)

In [4]:
margin = 60
number_of_samples =15
mtcnn= MTCNN(image_size=224, margin=margin, min_face_size= 30, thresholds=[0.6, 0.7, 0.7], post_process=False,device=device)

In [5]:
def get_number_of_frames(file_path: str) -> int:
    probe = ff.probe(file_path)
    video_streams = [stream for stream in probe["streams"] if stream["codec_type"] == "video"]
    #width = video_streams[0]['coded_width']
    #height = video_streams[0]['coded_height']
    del probe
    return video_streams[0]['nb_frames']

In [6]:
def extract_audio_from_video(file_path: str) -> np.ndarray:
    inputfile = ff.input(file_path)#입력 파일 만들기
    out = inputfile.output('-', format='f32le', acodec='pcm_f32le', ac=1, ar='44100',loglevel="quiet")#추출할 오디오 데이터 형식 지정（字节数组）
    raw = out.run(capture_stdout=True)#오디오 파일 추출
    del inputfile, out
    return np.frombuffer(raw[0],np.float32)

### 전체 오디오 프레임

In [ ]:

def preprocess_audio_series(raw_data: np.ndarray) -> np.ndarray:
    N, M = 24, 2585
    mfcc_data = librosa.feature.mfcc(y=raw_data, sr=44100, n_mfcc=24)#计算raw_data的MFCC特征 ,  n_mfcc=24  Mel Frequency Cepstrum Coefficient
    mfcc_data_standardized = (mfcc_data - np.mean(mfcc_data)) / np.std(mfcc_data)#mfcc_data를 표준화하다
    # Use pre-padding (Note: with 0, which is also the mean after standardization) to unify the length of the samples.
    number_of_columns_to_fill = M - mfcc_data_standardized.shape[1]#0을 몇 줄 더 보충해야 합니다.
    padding = np.zeros((N, number_of_columns_to_fill))
    padded_data = np.hstack((padding, mfcc_data_standardized))#배열이 수평으로 쌓입니다
   
    return padded_data.reshape(N, M, 1) # Reshaping to N,M,1
    #给定的(N, M, 1)形状可以看作是在指定时间内检测到的M个频率的强度值，其中N是MFCC特征数

### 전체 오디오 프레임 중에서 등간격으로 15프레임을 추출

In [8]:
def preprocess_audio_series_segmented(raw_data: np.ndarray, num_segments: int = 15) -> np.ndarray:    
    sr = 44100
    mfcc = librosa.feature.mfcc(y=raw_data, sr=sr, n_mfcc=24)
    mfcc = (mfcc - np.mean(mfcc)) / np.std(mfcc)  # 标准化
    
    N, T = mfcc.shape  # N=24, T=总帧数
    if T < 15:
        # 如果帧数太少，不足15帧，进行padding
        pad_width = 15 - T
        mfcc = np.pad(mfcc, ((0, 0), (0, pad_width)), mode='edge')
        T = mfcc.shape[1]

    # 等间隔抽取索引
    indices = np.linspace(0, T - 1, num=15, dtype=int)  # 长度15，范围[0, T-1]

    # 按帧抽取对应的MFCC列（shape: 24 × 15）
    sampled = mfcc[:, indices]  # shape: (24, 15)

    # 转换为 (15, 24, 1) 格式
    result = sampled.T[:, :, np.newaxis]  # (15, 24, 1)
    return result.astype(np.float32)

### 전체 오디오 프레임 15개의 시간 구간으로 나눔
### 구간별 mean pooling 적용된 후,하나의 프레임 생성 

In [9]:
# def preprocess_audio_series_segmented(raw_data: np.ndarray, num_segments: int = 15) -> np.ndarray:
#     sr = 44100
#     mfcc = librosa.feature.mfcc(y=raw_data, sr=sr, n_mfcc=24)
#     mfcc = (mfcc - np.mean(mfcc)) / np.std(mfcc)
    
#     # T: 时间帧长度
#     N, T = mfcc.shape
#     segment_len = T // num_segments  # 每个segment有多少帧
    
#     segmented = []
#     for i in range(num_segments):
#         start = i * segment_len
#         end = (i + 1) * segment_len if i < num_segments - 1 else T
#         segment = mfcc[:, start:end]
#         pooled = np.mean(segment, axis=1)  # shape: (24,)
#         segmented.append(pooled)

#     result = np.stack(segmented, axis=0)  # shape: (15, 24)
#     return result.astype(np.float32).reshape(num_segments, 24, 1)  # shape: (15, 24, 1)


In [10]:
#추출 할수 있는 얼굴
def extract_N_video_frames(frame_num: str ,file_path: str, number_of_samples: int = 15) -> List[np.ndarray]:
    full_video_frames = []#프레임 데이터를 저장할 배열 만들기
    face_video_frames = []
    max_attempt =  5
    frame_num=number_of_samples #추출된 프레임의 개수
    begin_num=0 # 프레임의 시작index
    indexes=[None] * number_of_samples # [0,2.7,5.4......]->[0,2,5......] // [0, 1, 2, 3, 4, 5.... 450?]
    get_frames=int(get_number_of_frames(file_path))
    x=get_frames/number_of_samples
    
    for i in range(frame_num):
        indexes[i]=int(begin_num)
        # print(type(get_number_of_frames(file_path)))
        begin_num+=x
    cap = cv2.VideoCapture(file_path)
    for t_idx, ind in enumerate(indexes):
        i=ind
        cap.set(1, ind)
        res, frame = cap.read()#프레임을 읽습니다
        # print("framedatatype",type(frame))
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        full_video_frames.append(frame)#RGB 형식으로 변환하여 video_frames에 추가합니다
    cap.release()
    del cap, indexes
    return full_video_frames


In [11]:
def resize_image(image: np.ndarray, new_size: Tuple[int,int]) -> np.ndarray:
    image = np.array(image)
    return cv2.resize(image, new_size, interpolation = cv2.INTER_AREA)
#在训练模型时，选取图像的一个随机128x128的窗口；在预测模型时，选取图像中心的128x128窗口。最后返回裁剪后的图像

In [12]:
def crop_image_window(image: np.ndarray, training: bool = True) -> np.ndarray:#这个不是太理解·······························
    height, width, _ = image.shape
    if training:
        MAX_N = height - 128
        MAX_M = width - 128
        rand_N_index, rand_M_index = random.randint(0, MAX_N) , random.randint(0, MAX_M)
        return image[rand_N_index:(rand_N_index+128),rand_M_index:(rand_M_index+128),:]
    else:
        N_index = (height - 128) // 2
        M_index = (width - 128) // 2
        return image[N_index:(N_index+128),M_index:(M_index+128),:]

In [13]:
def extract_third_character(filename):
    return filename[:3]
def reading_label_data(file_name: str, dictionary: Dict[str,str]) -> np.ndarray:#영상의 5대 점수를 획득했습니다
    index = int(extract_third_character(file_name)) 
    gt = dictionary.get(index, "ID not found")
    features = ['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']
    extracted_data = [float(gt[label]) for label in features]
    return np.stack(extracted_data).reshape(5,1)

In [14]:
def find_script_by_id(transcriptions_filepath,file_name):
    df = pd.read_csv(filepath_or_buffer=transcriptions_filepath, encoding="utf-8",dtype={'pid': str})
    filtered = df[(df['pid'] == file_name[:3]) & (df['SCENE'] == file_name[16:21])]
    # print(df['pid'][0])
    # print(df['SCENE'][0])
    # print(file_name[:3])
    # print(file_name[16:21])
    if not filtered.empty:
        return filtered['script'].values[0]
    else:
        logger.debug(file_name)
        logger.debug('ERROR. No Exist Script')
        return None

In [15]:
def preprocessing_input(file_path: str, file_name: str,transcriptions_filepath:str, dictionary: Dict[str, str]) -> Tuple[
    np.ndarray, np.ndarray, np.ndarray]:
    logger.debug(file_path)
    transcriptions = find_script_by_id(transcriptions_filepath, file_name)
    if transcriptions is None:
        logger.debug("transcriptions is None")
        return (None, None, None, None)
 
    # Audio
    extracted_audio_raw = extract_audio_from_video(file_path=file_path)#오디오 데이터 가져오기
    preprocessed_audio = preprocess_audio_series_segmented(raw_data=extracted_audio_raw,num_segments=number_of_samples)#오디오 파일 처리
    
    
    frame_num = int(get_number_of_frames(file_path))
    # Video
    full_video_frames = extract_N_video_frames(frame_num=frame_num,file_path=file_path, number_of_samples=number_of_samples)#128프레임 획득
    # > STYLE TRANSFER
    resized_Full_images = [resize_image(image=im, new_size=(224, 224)) for im in full_video_frames]#크기를 조정하다
    # cropped_images = [crop_image_window(image=resi, training=training) / 255.0 for resi in resized_images]#对图像进行裁剪
    if len(full_video_frames)==number_of_samples:
        preprocessed_full_video = np.stack(resized_Full_images)#하나의 배열을 합성하다
        video_gt = reading_label_data(file_name=file_name, dictionary=dictionary)#태그 읽기
        del full_video_frames
        return (preprocessed_audio,preprocessed_full_video,transcriptions, video_gt)
    else:
        logger.debug("frames not enough")
    return (None,None,None,None)

In [16]:
def reshape_to_expected_fullinput(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    x2_list = []
    x3_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
        x2_list.append(dataset[i][2])
        x3_list.append(dataset[i][3])
    return (np.stack(x0_list), np.stack(x1_list),np.stack(x2_list),np.stack(x3_list))

### TEST data extraction

In [ ]:
training_set_data = []
allsavename = '/ssd_data/extract_kr_15frame_v_a/extract_15video_15audio_data/train_set.dat'#
path ='/ssd_data/qx/KETIdata_by_scenario/speaking_only/divided_by_person/TEST2/train30s'#
excel_path = '/ssd_data/qx/KETIdata/keti_corpus_labels.csv'
transcriptions_filepath = '/ssd_data/qx/KETIdata/merged_transcriptions_single_line_per_file.csv'
df = pd.read_csv(excel_path)
gt = df[['id', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']]#id,recording_date,recording_id,gender,neuroticism,extraversion,openness,agreeableness,conscientiousness,neuroticism_60,extraversion_60,openness_60,agreeab
gt = pd.DataFrame(gt)
gt = gt.set_index('id')[['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']].to_dict(orient='index')
t1 = datetime.datetime.utcnow()
i=1
filenum=1 #500개의 비디오마다 하나의 파일에 저장
for filename in tqdm(os.listdir(path)):#파일 내 비디오 둘러보기
    filePath = path+'/'+filename
    data = preprocessing_input(file_path= filePath, file_name= filename, transcriptions_filepath=transcriptions_filepath , dictionary= gt)
    if int(get_number_of_frames(filePath))>128:
        if data[0] is None:
            continue
        else: 
            training_set_data.append(data)
            
            logger.debug('%s', i)
            i+=1
    
import gc
gc.collect() 
training_set_data = reshape_to_expected_fullinput(training_set_data)
with open(allsavename, "wb") as f:
    pickle.dump(training_set_data, f,protocol=4)  
fullsavename = []
t2 = datetime.datetime.utcnow()
#Measuring execution time
print('Elapsed time: ' + str(t2-t1))#
# gc.collect()
#[2025-04-18 02:36:11,819]::1689483::DEBUG::/ssd_data/qx/KETIdata/random_train/184_184_D019_S03_L4_SCENE3_018.mp4
# [2025-04-18 02:36:12,048]::1689483::DEBUG::29739

In [ ]:
training_set_data = []
allsavename = '/ssd_data/extract_kr_15frame_v_a/extract_15video_15audio_data/valid_set.dat'#
path ='/ssd_data/qx/KETIdata_by_scenario/speaking_only/divided_by_person/TEST2/valid30s'#
excel_path = '/ssd_data/qx/KETIdata/keti_corpus_labels.csv'
transcriptions_filepath = '/ssd_data/qx/KETIdata/merged_transcriptions_single_line_per_file.csv'
df = pd.read_csv(excel_path)
gt = df[['id', 'openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']]#id,recording_date,recording_id,gender,neuroticism,extraversion,openness,agreeableness,conscientiousness,neuroticism_60,extraversion_60,openness_60,agreeab
gt = pd.DataFrame(gt)
gt = gt.set_index('id')[['openness', 'conscientiousness', 'extraversion', 'agreeableness', 'neuroticism']].to_dict(orient='index')
t1 = datetime.datetime.utcnow()
i=1
filenum=1 #500개의 비디오마다 하나의 파일에 저장
for filename in tqdm(os.listdir(path)):#파일 내 비디오 둘러보기
    filePath = path+'/'+filename
    data = preprocessing_input(file_path= filePath, file_name= filename, transcriptions_filepath=transcriptions_filepath , dictionary= gt)
    if int(get_number_of_frames(filePath))>128:
        if data[0] is None:
            continue
        else: 
            training_set_data.append(data)
            
            logger.debug('%s', i)
            i+=1
    
import gc
gc.collect() 
training_set_data = reshape_to_expected_fullinput(training_set_data)
with open(allsavename, "wb") as f:
    pickle.dump(training_set_data, f,protocol=4)  
fullsavename = []
t2 = datetime.datetime.utcnow()
#Measuring execution time
print('Elapsed time: ' + str(t2-t1))#
# gc.collect()
#[2025-04-18 02:36:11,819]::1689483::DEBUG::/ssd_data/qx/KETIdata/random_train/184_184_D019_S03_L4_SCENE3_018.mp4
# [2025-04-18 02:36:12,048]::1689483::DEBUG::29739

  2%|▏         | 13/535 [00:04<02:30,  3.48it/s]